# Same-lobe-pair CSE pilot

This notebook is a separate version of the xyz-observation/xyz-control Qiwen
batch. It writes only to `xyz_control_same_lobe_pair_qiwenevents`; the existing
`sign_invariant` notebook and results remain untouched. Run the cells in order.


In [ ]:
from project_paths import (
    PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR,
    cse_experiment_paths, ensure_output_directories,
)
ensure_output_directories()
EXPERIMENT_NAME = 'xyz_control_same_lobe_pair_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, EXPERIMENT_FIGURES_DIR = cse_experiment_paths(EXPERIMENT_NAME)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# Qiwen CSE xyz control — Same-lobe safe–crossing pairing


The only scientific change from the existing `sign_invariant` CSE is in
`past_safe_control`. When all current ensemble members forecast a crossing,
choose one crossing member first. A past candidate is safe only if its aligned
forecast stays strictly in **that member's starting lobe**. The sign comes
from the analysis-member forecast, never from the controlled nature run.
The old `sign_invariant` mode is preserved unchanged in the shared CSE code.

This pilot uses the same xyz OSSE, Qiwen event catalogue, 40 event indices,
seeds `1000 + event_index`, `D=0.05`, and warning horizon `T=4T0` as the
previous grid point. It saves each event separately, so an interrupted run
can resume without rewriting completed events. The 40-event pilot is followed by a separate 5×9 T–D grid. The grid is not executed automatically and saves each event separately; the [4T0], D=0.05 point reuses the pilot files. The previously saved `sign_invariant` rate is the comparator.


## Imports


In [ ]:
import importlib
import pickle
from pathlib import Path

import numpy as np
from class_da_sys import da_system
import class_cse_qiwen as cse_qiwen

cse_qiwen = importlib.reload(cse_qiwen)
detect_regime_changes = cse_qiwen.detect_regime_changes
lorenz63 = cse_qiwen.lorenz63
run_single_cse_core = cse_qiwen.run_single_cse


## DA System Setup


In [ ]:
BASE_PATH = PROJECT_ROOT
PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)

# This notebook assimilates x, y, z and applies CSE perturbations to x, y, and z.
# control_indices=None makes build_and_apply_perturbation skip component masking
# and perturb x, y, and z, which is the original CSE case in Figure 3(a) of the paper.
CONTROL_INDICES = None
CONTROL_LABEL = 'xyz'
REGIME_REFERENCE_MODE = 'same_lobe_pair'
AMBIGUOUS_REGIME_POLICY = 'continue'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04


## Load OSSE Histories


In [ ]:
with open(OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
# Detect every regime transition in the NATURE trajectory as a sign change
# of the x component (a crossing of x=0 between two consecutive time steps).
# With EVENT_SELECTION='qiwen', the function accepts crossings in both
# directions, keeps those occurring after step 500 whose preceding residence
# time is longer than 385 steps, and retains at most the first 100 events.
# The complete list of crossings is also saved in RESULTS_DIR for reuse.
regime_change_info = detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='cse_qiwen_xyz_control_change_region_time_qiwen_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

# All detected x=0 crossings in the NATURE run.
change_region_time = regime_change_info['change_region_time']
# Forecast/control horizon used by the function's diagnostic sign check.
check_position = regime_change_info['check_position']
# Filtered crossing times used as the CSE event set.
t_start_series = regime_change_info['t_start_series']
# Time spent in the previous regime before each selected crossing.
t_start_residence = regime_change_info['t_start_residence']
# Mean interval between all detected crossings, before event filtering.
mean_crossing_interval = regime_change_info['mean_crossing_interval']


## Batch Parameters

Se conservan los mismos parámetros del experimento individual. `experiment_step` limita cada realización y `position` proporciona el horizonte fijo de 4T0 usado en el primer batch.


In [ ]:
# Keep the shared CSE defaults consistent with the individual-event notebook.
# The batch cells replace D and EVENT_INDEX with their own D_run and event index;
# experiment_step remains the maximum number of DA cycles per realization.
D = 0.05
amp = 1.0 / D
experiment_step = 1000
EVENT_INDEX = 1

# Lorenz-63 parameters and integration time step used by the CSE.
solver_params = {
    'model_func': lorenz63,
    'sigma': 10.0,
    'beta': 8.0 / 3.0,
    'rho': 28.0,
    'dt': fcst_dt,
}

print('position:', position)


## Reusable Single-Event Runner

Esta envoltura conecta la configuración del notebook con `run_single_cse_core`. No modifica ni reduce el resultado devuelto.


In [ ]:
# Thin notebook wrapper around the single authoritative implementation.
# The complete result dictionary is returned unchanged so batch pickle files
# retain scalar metrics, trajectories, assimilation histories, branch choices,
# observations, perturbation times, and ensemble forecast diagnostics.
def run_single_cse(event_index, D_run, seed, regime_reference='same_lobe_pair', candidate_pool='full', cle_target_output_dir=None):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        ambiguous_regime_policy=AMBIGUOUS_REGIME_POLICY,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_number,
        cle_target_output_dir=cle_target_output_dir,
    )


## Run the 40-event pilot

Only this new result directory is written. Each event is saved atomically in
its own pickle, with the mode and configuration checked before resuming.


In [ ]:
import os
import tempfile
import time

PILOT_D = 0.05
PILOT_D_TAG = '0p05'
N_PILOT_EVENTS = 40
assert REGIME_REFERENCE_MODE == 'same_lobe_pair'
assert CONTROL_INDICES is None
assert multiple_number == 4
assert len(t_start_series) >= N_PILOT_EVENTS


def pilot_result_path(event_index):
    return RESULTS_DIR / (
        f'cse_qiwen_xyz_control_same_lobe_pair_event_{event_index:02d}'
        f'_D{PILOT_D_TAG}_4T0_qiwenevents.pkl'
    )


def check_pilot_result(row, event_index):
    expected = {
        'event_index': event_index,
        'D': PILOT_D,
        'regime_reference': REGIME_REFERENCE_MODE,
        'candidate_pool': 'full',
        'control_indices': None,
        't_start': t_start_series[event_index],
    }
    for key, value in expected.items():
        if row.get(key) != value:
            raise ValueError(
                f'Existing event {event_index} has {key}={row.get(key)!r}, '
                f'expected {value!r}; refusing to reuse it.'
            )
    summary = row.get('cse_summary', {})
    for key, value in {
        'position': position,
        'seed': 1000 + event_index,
        'requested_num_cycles': experiment_step,
    }.items():
        if summary.get(key) != value:
            raise ValueError(
                f'Existing event {event_index} has summary {key}='
                f'{summary.get(key)!r}, expected {value!r}; '
                'refusing to reuse it.'
            )


pilot_rows = []
for event_index in range(N_PILOT_EVENTS):
    result_path = pilot_result_path(event_index)
    if result_path.exists():
        with result_path.open('rb') as stream:
            row = pickle.load(stream)
        status = 'reused'
    else:
        started = time.time()
        row = run_single_cse(
            event_index, PILOT_D, seed=1000 + event_index,
            regime_reference=REGIME_REFERENCE_MODE,
            candidate_pool='full',
        )
        check_pilot_result(row, event_index)
        with tempfile.NamedTemporaryFile(
            mode='wb', prefix=result_path.stem + '_', suffix='.tmp',
            dir=RESULTS_DIR, delete=False,
        ) as stream:
            pickle.dump(row, stream, protocol=pickle.HIGHEST_PROTOCOL)
            temporary_path = Path(stream.name)
        os.replace(temporary_path, result_path)
        status = f'computed in {(time.time() - started) / 60:.1f} min'
    check_pilot_result(row, event_index)
    pilot_rows.append({
        'event_index': event_index,
        'success': bool(row['success']),
        'failure_cycle': row['failure_cycle'],
        'failure_reason': row.get('failure_reason'),
        'control_ratio': float(row['control_ratio']),
        'n_past_safe': int(row['branch_counts']['past_safe_control']),
    })
    print(
        f'[{event_index + 1}/{N_PILOT_EVENTS}] '
        f'success={row["success"]} '
        f'past_safe={row["branch_counts"]["past_safe_control"]} '
        f'({status})'
    )
    del row


## Pilot success rate and saved-reference comparison

This cell only reads saved event files and the earlier aggregate CSV. It does
not run CSE. The old `nature` setting is not used as a publication baseline.


In [ ]:
import csv

event_summaries = []
for event_index in range(N_PILOT_EVENTS):
    result_path = pilot_result_path(event_index)
    if not result_path.is_file():
        print(f'Missing paired event: {event_index}')
        continue
    with result_path.open('rb') as stream:
        row = pickle.load(stream)
    check_pilot_result(row, event_index)
    event_summaries.append({
        'event': event_index,
        'success': bool(row['success']),
        'failure_cycle': row['failure_cycle'],
        'failure_reason': row.get('failure_reason'),
        'past_safe_cycles': int(row['branch_counts']['past_safe_control']),
        'control_ratio': float(row['control_ratio']),
    })
    del row

n_done = len(event_summaries)
n_success = sum(item['success'] for item in event_summaries)
print(f'Same-lobe pair: {n_success}/{n_done} successful '
      f'({100 * n_success / n_done:.1f}%)' if n_done else 'No paired events saved.')
print('Event  Success  Failure cycle  Reason')
for item in event_summaries:
    print(f"{item['event']:>5}  {str(item['success']):>7}  "
          f"{str(item['failure_cycle']):>13}  {item['failure_reason']}")

baseline_csv = (
    PROJECT_ROOT / 'outputs' / '05_cse_qiwen'
    / 'xyz_control_sign_invariant_qiwenevents' / 'results'
    / 'cse_qiwen_xyz_control_40_fig3a_grid_aggregates_sign_invariant_qiwenevents.csv'
)
if baseline_csv.is_file():
    with baseline_csv.open(newline='') as stream:
        baseline_rows = list(csv.DictReader(stream))
    matching = [
        item for item in baseline_rows
        if item['T'] == '[4T0]' and float(item['D']) == PILOT_D
    ]
    if len(matching) == 1:
        baseline = matching[0]
        print('Saved sign_invariant baseline: '
              f"{baseline['success_ratio']} success ratio "
              f"over {baseline['n_events']} events")
else:
    print('Sign-invariant aggregate CSV not found; paired results remain valid.')


## Batch Run: Figure 3(a) T-D Grid


In [ ]:
# Full Figure 3(a) grid: five warning horizons, nine D values, 40 events.
# This is expensive (up to 1,800 CSE runs and large diagnostic pickles).
# Results are isolated in the same_lobe_pair experiment directory. An
# interrupted run resumes from saved events without rerunning them.
import os
import pickle
import tempfile
import time

FIG3A_T_MULTIPLES = {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
}
FIG3A_D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
FIG3A_N_EVENTS = 40
FIG3A_REGIME_REFERENCE = 'same_lobe_pair'
FIG3A_CANDIDATE_POOL = 'full'
assert REGIME_REFERENCE_MODE == FIG3A_REGIME_REFERENCE
assert CONTROL_INDICES is None
assert len(t_start_series) >= FIG3A_N_EVENTS


def fig3a_event_path(T_label, D_run, event_index):
    # Reuse the independently saved pilot at this exact grid point.
    if T_label == '[4T0]' and D_run == 0.05:
        return RESULTS_DIR / (
            f'cse_qiwen_xyz_control_same_lobe_pair_event_{event_index:02d}'
            '_D0p05_4T0_qiwenevents.pkl'
        )
    t_tag = T_label.strip('[]').replace('.', 'p')
    d_tag = f'{D_run:g}'.replace('.', 'p')
    return RESULTS_DIR / (
        f'cse_qiwen_xyz_control_same_lobe_pair_fig3a_{t_tag}'
        f'_D{d_tag}_event_{event_index:02d}{OUTPUT_TAG}.pkl'
    )


def check_fig3a_event(row, T_label, D_run, event_index):
    multiple_run = FIG3A_T_MULTIPLES[T_label]
    position_run = int(np.ceil(T_0 * multiple_run))
    expected = {
        'event_index': event_index,
        'D': D_run,
        'regime_reference': FIG3A_REGIME_REFERENCE,
        'candidate_pool': FIG3A_CANDIDATE_POOL,
        'control_indices': None,
        't_start': t_start_series[event_index],
    }
    for key, value in expected.items():
        if row.get(key) != value:
            raise ValueError(
                f'{fig3a_event_path(T_label, D_run, event_index)}: '
                f'{key}={row.get(key)!r}, expected {value!r}'
            )
    summary = row.get('cse_summary', {})
    for key, value in {
        'position': position_run,
        'seed': 1000 + event_index,
        'requested_num_cycles': experiment_step,
    }.items():
        if summary.get(key) != value:
            raise ValueError(
                f'{fig3a_event_path(T_label, D_run, event_index)}: '
                f'summary {key}={summary.get(key)!r}, expected {value!r}'
            )


for T_label, multiple_run in FIG3A_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    print(f'=== T={T_label}, position={position_run} ===')
    for D_run in FIG3A_D_VALUES:
        for event_index in range(FIG3A_N_EVENTS):
            result_path = fig3a_event_path(T_label, D_run, event_index)
            if result_path.is_file():
                with result_path.open('rb') as stream:
                    row = pickle.load(stream)
                status = 'reused'
            else:
                started = time.time()
                row = run_single_cse_core(
                    event_index=event_index,
                    D_run=D_run,
                    seed=1000 + event_index,
                    das=das,
                    x_nature=x_nature,
                    t_start_series=t_start_series,
                    osse_Xa_history=osse_Xa_history,
                    osse_summary=osse_summary,
                    experiment_step=experiment_step,
                    position=position_run,
                    Ta=Ta,
                    fcst_dt=fcst_dt,
                    path_figures=PathFigures,
                    total_events=FIG3A_N_EVENTS,
                    regime_reference=FIG3A_REGIME_REFERENCE,
                    candidate_pool=FIG3A_CANDIDATE_POOL,
                    ambiguous_regime_policy=AMBIGUOUS_REGIME_POLICY,
                    control_indices=CONTROL_INDICES,
                    T_0=T_0,
                    multiple_number=multiple_run,
                )
                check_fig3a_event(row, T_label, D_run, event_index)
                with tempfile.NamedTemporaryFile(
                    mode='wb', prefix=result_path.stem + '_', suffix='.tmp',
                    dir=RESULTS_DIR, delete=False,
                ) as stream:
                    pickle.dump(row, stream, protocol=pickle.HIGHEST_PROTOCOL)
                    temporary_path = Path(stream.name)
                os.replace(temporary_path, result_path)
                status = f'computed in {(time.time() - started) / 60:.1f} min'
            check_fig3a_event(row, T_label, D_run, event_index)
            print(
                f'T={T_label}, D={D_run:g}, '
                f'event={event_index + 1}/{FIG3A_N_EVENTS}, '
                f'success={row["success"]} ({status})'
            )
            del row


## Figure 3(a) Grid Summary and Comparison

The Qiwen table is a historical benchmark. The same-lobe-pair results are
separate from the previous `sign_invariant` results. Cells with fewer than
40 saved events are preliminary and show their event count.


In [ ]:
# Post-process saved same-lobe Figure 3(a) event files only.
# Missing and partial grid points remain blank in the success matrix; the
# aggregate table still shows each partial point's event count and rate.
# This cell does not run CSE or touch the sign_invariant outputs.
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG3A_T_MULTIPLES = globals().get('FIG3A_T_MULTIPLES', {
    '[1.5T0]': 1.5, '[2T0]': 2.0, '[2.5T0]': 2.5,
    '[3T0]': 3.0, '[4T0]': 4.0,
})
FIG3A_D_VALUES = globals().get(
    'FIG3A_D_VALUES', [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
)
FIG3A_N_EVENTS = globals().get('FIG3A_N_EVENTS', 40)

# Historical values transcribed in the existing sign_invariant notebook.
QIWEN_FIG3A_CONTROL_XYZ_VALUES = {
    '[1.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.0, 0.3: 0.0, 0.4: 0.0, 0.5: 0.025},
    '[2T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.025, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[2.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.025, 0.1: 0.250, 0.2: 0.275, 0.3: 0.250, 0.4: 0.200, 0.5: 0.175},
    '[3T0]': {0.02: 0.0, 0.03: 0.050, 0.04: 0.425, 0.05: 0.550, 0.1: 0.800, 0.2: 0.675, 0.3: 0.400, 0.4: 0.100, 0.5: 0.200},
    '[4T0]': {0.02: 0.050, 0.03: 0.975, 0.04: 0.925, 0.05: 0.975, 0.1: 0.825, 0.2: 0.825, 0.3: 0.725, 0.4: 0.500, 0.5: 0.525},
}


def fig3a_event_path(T_label, D_run, event_index):
    if T_label == '[4T0]' and D_run == 0.05:
        return RESULTS_DIR / (
            f'cse_qiwen_xyz_control_same_lobe_pair_event_{event_index:02d}'
            '_D0p05_4T0_qiwenevents.pkl'
        )
    t_tag = T_label.strip('[]').replace('.', 'p')
    d_tag = f'{D_run:g}'.replace('.', 'p')
    return RESULTS_DIR / (
        f'cse_qiwen_xyz_control_same_lobe_pair_fig3a_{t_tag}'
        f'_D{d_tag}_event_{event_index:02d}{OUTPUT_TAG}.pkl'
    )


event_rows = []
aggregate_rows = []
for T_label, multiple_run in FIG3A_T_MULTIPLES.items():
    for D_run in FIG3A_D_VALUES:
        rows = []
        for event_index in range(FIG3A_N_EVENTS):
            result_path = fig3a_event_path(T_label, D_run, event_index)
            if not result_path.is_file():
                continue
            with result_path.open('rb') as stream:
                row = pickle.load(stream)
            if (row.get('event_index') != event_index
                    or row.get('D') != D_run
                    or row.get('regime_reference') != 'same_lobe_pair'
                    or row.get('cse_summary', {}).get('position')
                    != int(np.ceil(T_0 * multiple_run))):
                raise ValueError(f'Unexpected configuration in {result_path}')
            event_rows.append({
                'T': T_label, 'multiple_number': multiple_run,
                'position': int(np.ceil(T_0 * multiple_run)),
                'D': D_run, 'event': event_index,
                'success': bool(row['success']),
                'control_ratio': row['control_ratio'],
                'false_alarm_ratio': row['false_alarm_ratio'],
                'success_control_ratio': row['success_control_ratio'],
                'failure_cycle': row['failure_cycle'],
                'result_file': str(result_path),
            })
            rows.append({
                'success': bool(row['success']),
                'control_ratio': row['control_ratio'],
                'false_alarm_ratio': row['false_alarm_ratio'],
                'success_control_ratio': row['success_control_ratio'],
            })
            del row
        n_events = len(rows)
        aggregate_rows.append({
            'T': T_label, 'multiple_number': multiple_run,
            'position': int(np.ceil(T_0 * multiple_run)), 'D': D_run,
            'n_events': n_events,
            'success_ratio': float(np.mean([r['success'] for r in rows])) if rows else np.nan,
            'mean_control_ratio': float(np.nanmean([r['control_ratio'] for r in rows])) if rows else np.nan,
            'mean_false_alarm_ratio': float(np.nanmean([r['false_alarm_ratio'] for r in rows])) if rows else np.nan,
            'mean_success_control_ratio': float(np.nanmean([r['success_control_ratio'] for r in rows])) if rows else np.nan,
            'n_failures': sum(not r['success'] for r in rows) if rows else np.nan,
        })
        del rows

fig3g_event_table = pd.DataFrame(event_rows)
fig3g_aggregate_table = pd.DataFrame(aggregate_rows)
fig3g_aggregate_table['complete_success_ratio'] = fig3g_aggregate_table[
    'success_ratio'
].where(fig3g_aggregate_table['n_events'] == FIG3A_N_EVENTS)
fig3g_success_matrix = fig3g_aggregate_table.pivot(
    index='T', columns='D', values='complete_success_ratio'
).reindex(index=list(FIG3A_T_MULTIPLES), columns=FIG3A_D_VALUES)
qiwen_fig3g_matrix = pd.DataFrame.from_dict(
    QIWEN_FIG3A_CONTROL_XYZ_VALUES, orient='index'
).reindex(index=list(FIG3A_T_MULTIPLES), columns=FIG3A_D_VALUES)
fig3g_difference_matrix = fig3g_success_matrix - qiwen_fig3g_matrix

fig3g_event_table_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_grid_events_same_lobe_pair_qiwenevents.csv'
fig3g_aggregate_table_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_grid_aggregates_same_lobe_pair_qiwenevents.csv'
fig3g_success_matrix_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_success_matrix_same_lobe_pair_qiwenevents.csv'
fig3g_difference_matrix_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_minus_qiwen_matrix_same_lobe_pair_qiwenevents.csv'
fig3g_event_table.to_csv(fig3g_event_table_file, index=False)
fig3g_aggregate_table.to_csv(fig3g_aggregate_table_file, index=False)
fig3g_success_matrix.to_csv(fig3g_success_matrix_file)
fig3g_difference_matrix.to_csv(fig3g_difference_matrix_file)

display(fig3g_aggregate_table[['T', 'D', 'n_events', 'success_ratio']])
display(fig3g_success_matrix)
display(fig3g_difference_matrix)
print('Only cells with n_events=40 are complete 40-event success rates.')

PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)
PAPER_TABLE_COLORS = {
    'ge_08': '#9c7a00',
    'ge_06': '#d1a000',
    'ge_04': '#ffe173',
    'ge_02': '#fff0b3',
    'gt_00': '#fff8dc',
    'zero': '#ffffff',
    'missing': '#f2f2f2',
}


def _paper_cell_color(value):
    if pd.isna(value):
        return PAPER_TABLE_COLORS['missing']
    value = float(value)
    if np.isclose(value, 0.0):
        return PAPER_TABLE_COLORS['zero']
    if value >= 0.8:
        return PAPER_TABLE_COLORS['ge_08']
    if value >= 0.6:
        return PAPER_TABLE_COLORS['ge_06']
    if value >= 0.4:
        return PAPER_TABLE_COLORS['ge_04']
    if value >= 0.2:
        return PAPER_TABLE_COLORS['ge_02']
    return PAPER_TABLE_COLORS['gt_00']


def _format_paper_value(value):
    if pd.isna(value):
        return ''
    value = float(value)
    if np.isclose(value, 0.0):
        return '0'
    return f'{value:.3f}'.rstrip('0').rstrip('.')


def _format_D_label(value):
    return f'{float(value):.2f}'.rstrip('0').rstrip('.')


def _draw_paper_table(ax, matrix, title):
    ax.axis('off')
    header = [r'T \\ D'] + [_format_D_label(D_run) for D_run in matrix.columns]
    table_text = [header]
    table_colors = [['#ffffff'] * len(header)]

    for T_label in matrix.index:
        row_text = [T_label]
        row_colors = ['#ffffff']
        for D_run in matrix.columns:
            value = matrix.loc[T_label, D_run]
            row_text.append(_format_paper_value(value))
            row_colors.append(_paper_cell_color(value))
        table_text.append(row_text)
        table_colors.append(row_colors)

    mpl_table = ax.table(
        cellText=table_text,
        cellColours=table_colors,
        cellLoc='center',
        loc='center',
        bbox=[0.0, 0.0, 1.0, 0.88],
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8.4)
    mpl_table.scale(1.0, 1.16)

    for (row, col), table_cell in mpl_table.get_celld().items():
        table_cell.set_edgecolor('black')
        table_cell.set_linewidth(0.65)
        if row == 0 or col == 0:
            table_cell.set_text_props(weight='normal', color='black')
            table_cell.set_facecolor('#ffffff')
        elif table_cell.get_facecolor()[:3] == (0.611764705882353, 0.47843137254901963, 0.0):
            table_cell.set_text_props(color='white')

    ax.text(0.5, 0.945, title, ha='center', va='center', fontsize=10)
    return mpl_table


fig = plt.figure(figsize=(11.8, 4.5), constrained_layout=False)
grid = fig.add_gridspec(2, 2, height_ratios=[0.22, 1.0], hspace=0.12, wspace=0.08)
legend_ax = fig.add_subplot(grid[0, :])
legend_ax.axis('off')
legend_labels = [[r'$\geq 0.8$', r'$\geq 0.6$', r'$\geq 0.4$', r'$\geq 0.2$', r'$> 0$', r'$0$']]
legend_colors = [[
    PAPER_TABLE_COLORS['ge_08'],
    PAPER_TABLE_COLORS['ge_06'],
    PAPER_TABLE_COLORS['ge_04'],
    PAPER_TABLE_COLORS['ge_02'],
    PAPER_TABLE_COLORS['gt_00'],
    PAPER_TABLE_COLORS['zero'],
]]
legend_table = legend_ax.table(
    cellText=legend_labels,
    cellColours=legend_colors,
    cellLoc='center',
    loc='center',
    bbox=[0.38, 0.18, 0.24, 0.62],
)
legend_table.auto_set_font_size(False)
legend_table.set_fontsize(8.2)
for (row, col), table_cell in legend_table.get_celld().items():
    table_cell.set_edgecolor('black')
    table_cell.set_linewidth(0.65)
    if col == 0:
        table_cell.set_text_props(color='white')

axes = [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])]
_draw_paper_table(axes[0], qiwen_fig3g_matrix, '(a) Qiwen control xyz')
_draw_paper_table(axes[1], fig3g_success_matrix, '(a) Diego same-lobe pair xyz')

fig3g_grid_png = PathFigures / 'cse_qiwen_xyz_control_fig3a_grid_comparison_same_lobe_pair_qiwenevents.png'
fig.savefig(fig3g_grid_png, dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved event table:', fig3g_event_table_file)
print('Saved aggregate table:', fig3g_aggregate_table_file)
print('Saved success matrix:', fig3g_success_matrix_file)
print('Saved difference matrix:', fig3g_difference_matrix_file)
print('Saved comparison figure:', fig3g_grid_png)
